# 04 — Ba mô hình boosting mới có hơn HGB không?

**Giả thuyết:** CatBoost có thể tận dụng biến phân loại; LightGBM và XGBoost có khả năng học tương tác phi tuyến. Chạy trên **cùng split** với HGB và cùng tập feature ở mỗi cutoff. Đây là so sánh validation, chưa phải kết luận test.

Đây là nhóm ưu tiên trước học sâu vì dữ liệu đang ở dạng bảng. Tất cả phiên bản và tham số được ghi lại để tái lập.

In [1]:
from pathlib import Path
import sys
ROOT = Path.cwd()
if not (ROOT / 'clht_code').exists():
    ROOT = ROOT.parents[1]
sys.path.insert(0, str(ROOT))
from NghienCuu_HCO_VNLS.code_dung_chung.thuc_nghiem import (
    GOC, NGHIEN_CUU, DATA, KET_QUA, HINH, CAC_MOC,
    tao_thu_muc, duong_dan, chia_nhom, nap_du_lieu, chon_dac_trung,
    tien_xu_ly, bo_sung_ty_le, bo_sung_cua_so_vle,
)
tao_thu_muc()
print('Dữ liệu nguồn:', DATA)
print('Kết quả riêng:', KET_QUA)

Dữ liệu nguồn: CLHT_GB_GITHUB\data
Kết quả riêng: CLHT_GB_GITHUB\NghienCuu_HCO_VNLS\ket_qua


In [2]:
import time
import pandas as pd
from IPython.display import display
from sklearn.pipeline import Pipeline
from clht_code.evaluation import metric_table
from clht_code.models import ModelConfig, make_pipeline
from xgboost import XGBClassifier
import importlib.util
import xgboost
available = {'CatBoost': importlib.util.find_spec('catboost') is not None,
             'LightGBM': importlib.util.find_spec('lightgbm') is not None, 'XGBoost': True}
if available['CatBoost']:
    from catboost import CatBoostClassifier
if available['LightGBM']:
    from lightgbm import LGBMClassifier
print('Có thể chạy:', available, '| XGBoost', xgboost.__version__)

Có thể chạy: {'CatBoost': True, 'LightGBM': True, 'XGBoost': True} | XGBoost 3.1.2


## 1. Cùng dữ liệu, khác cách học

CatBoost nhận category gốc; những chỗ khuyết được đặt là `THIEU` (một nhãn category). LightGBM và XGBoost dùng median imputation/OneHotEncoder được fit **chỉ trên train**. Ba model dùng cấu hình ban đầu đơn giản, chưa tuning rộng; không được tuyên bố một model kém hẳn chỉ từ lần chạy này.

In [3]:
def fit_catboost(train, valid, features):
    X_train = train[features].copy()
    X_valid = valid[features].copy()
    categorical = X_train.select_dtypes(include=['object', 'category', 'bool']).columns.tolist()
    for column in categorical:
        X_train[column] = X_train[column].fillna('THIEU').astype(str)
        X_valid[column] = X_valid[column].fillna('THIEU').astype(str)
    model = CatBoostClassifier(iterations=240, depth=5, learning_rate=.055, l2_leaf_reg=5,
                               loss_function='Logloss', verbose=False, thread_count=4, random_seed=41,
                               allow_writing_files=False)
    model.fit(X_train, train.risk, cat_features=categorical)
    return model.predict_proba(X_valid)[:, 1]

def fit_other(name, train, valid, features):
    if name == 'LightGBM':
        estimator = LGBMClassifier(n_estimators=210, learning_rate=.045, num_leaves=15,
                                   min_child_samples=35, random_state=41, n_jobs=4, verbosity=-1)
    else:
        estimator = XGBClassifier(n_estimators=210, learning_rate=.045, max_depth=4,
                                  min_child_weight=6, subsample=.85, colsample_bytree=.85,
                                  tree_method='hist', random_state=41, n_jobs=4, eval_metric='logloss')
    pipeline = Pipeline([('preprocess', tien_xu_ly(train[features])), ('model', estimator)])
    pipeline.fit(train[features], train.risk)
    return pipeline.predict_proba(valid[features])[:, 1]

results = []
for cutoff in CAC_MOC:
    frame = nap_du_lieu(cutoff)
    features = chon_dac_trung(frame)
    train, valid = (frame[frame.split == name] for name in ('train', 'validation'))
    for name in ('CatBoost', 'LightGBM', 'XGBoost'):
        if not available[name]:
            print(f'{name}: chưa cài thư viện, không tạo kết quả giả')
            continue
        started = time.perf_counter()
        probability = fit_catboost(train, valid, features) if name == 'CatBoost' else fit_other(name, train, valid, features)
        metric = metric_table(valid.risk, probability, .5)
        results.append({'cutoff': cutoff, 'model': name, 'feature_count': len(features),
                        'seconds': time.perf_counter()-started, **metric})
        print(f'{cutoff:.0%} {name}: Accuracy={metric["accuracy"]:.4f}, AP={metric["average_precision_risk"]:.4f}')
        if cutoff in (.5, .75):
            pd.DataFrame({'id_student': valid.id_student, 'code_module': valid.code_module,
                          'code_presentation': valid.code_presentation, 'risk': valid.risk,
                          'probability': probability}).to_csv(
                KET_QUA/f'du_doan_validation_{name}_{round(cutoff*100)}.csv', index=False, encoding='utf-8-sig')
results = pd.DataFrame(results)
results.to_csv(KET_QUA/'boosting_validation.csv', index=False, encoding='utf-8-sig')
display(results[['cutoff','model','accuracy','precision_risk','recall_risk','f1_risk',
                 'weighted_f1','average_precision_risk','brier_score','seconds']].round(4))

50% CatBoost: Accuracy=0.8153, AP=0.8548


50% LightGBM: Accuracy=0.8153, AP=0.8526


50% XGBoost: Accuracy=0.8164, AP=0.8528


60% CatBoost: Accuracy=0.8448, AP=0.8776


60% LightGBM: Accuracy=0.8434, AP=0.8751


60% XGBoost: Accuracy=0.8397, AP=0.8752


... (6 outputs left out)

75% CatBoost: Accuracy=0.8750, AP=0.9019


75% LightGBM: Accuracy=0.8750, AP=0.8992


75% XGBoost: Accuracy=0.8779, AP=0.9002


,cutoff,model,accuracy,precision_risk,recall_risk,f1_risk,weighted_f1,average_precision_risk,brier_score,seconds
0,0.50,CatBoost,0.8153,0.8492,0.6281,0.7221,0.8083,0.8548,0.1294,16.4019
1,0.50,LightGBM,0.8153,0.8374,0.6410,0.7262,0.8093,0.8526,0.1306,0.4658
2,0.50,XGBoost,0.8164,0.8478,0.6331,0.7249,0.8097,0.8528,0.1305,0.5454
3,0.60,CatBoost,0.8448,0.8830,0.6618,0.7565,0.8389,0.8776,0.1122,17.6589
4,0.60,LightGBM,0.8434,0.8777,0.6625,0.7551,0.8376,0.8751,0.1132,0.4139
5,0.60,XGBoost,0.8397,0.8737,0.6548,0.7486,0.8336,0.8752,0.1132,0.5149
6,0.65,CatBoost,0.8580,0.8819,0.6913,0.7750,0.8534,0.8891,0.1028,16.2307
7,0.65,LightGBM,0.8592,0.8871,0.6897,0.7761,0.8544,0.8880,0.1032,0.4355
8,0.65,XGBoost,0.8557,0.8793,0.6864,0.7710,0.8509,0.8877,0.1034,0.5093
9,0.70,CatBoost,0.8653,0.8962,0.6926,0.7813,0.8605,0.8952,0.0979,16.1628


## 2. Đặt cạnh HGB đúng cùng protocol

Nếu mốc Accuracy 90% không đạt, không thay đổi threshold hoặc chọn test để khiến con số đẹp hơn. Kiểm tra đồng thời AP, Recall risk và số cảnh báo.

In [4]:
hgb = pd.read_csv(KET_QUA/'hgb_theo_cutoff_validation.csv')
compare = pd.concat([hgb, results], ignore_index=True)
display(compare.pivot(index='cutoff', columns='model', values=['accuracy','recall_risk','average_precision_risk']).round(4))
print('Bảng này là validation; không phải test-lock.')

accuracy                          recall_risk                           \
model  CatBoost HGB_160 LightGBM XGBoost    CatBoost HGB_160 LightGBM XGBoost   
cutoff                                                                          
0.50     0.8153  0.8134   0.8153  0.8164      0.6281  0.6439   0.6410  0.6331   
0.60     0.8448  0.8425   0.8434  0.8397      0.6618  0.6695   0.6625  0.6548   
0.65     0.8580  0.8560   0.8592  0.8557      0.6913  0.6913   0.6897  0.6864   
0.70     0.8653  0.8644   0.8639  0.8668      0.6926  0.6984   0.6884  0.6917   
0.75     0.8750  0.8773   0.8750  0.8779      0.7018  0.7044   0.7001  0.7078   

       average_precision_risk                           
model                CatBoost HGB_160 LightGBM XGBoost  
cutoff                                                  
0.50                   0.8548  0.8505   0.8526  0.8528  
0.60                   0.8776  0.8742   0.8751  0.8752  
0.65                   0.8891  0.8870   0.8880  0.8877  
0.70                   0.8952  0.8928   0.8937  0.8944  
0.75                   0.9019  0.8986   0.8992  0.9002

Bảng này là validation; không phải test-lock.


## Kết quả đọc nhanh — chạy lại 24/09/2026

CatBoost đã được cài và chạy đủ năm cutoff. Tại 50%, CatBoost đạt Accuracy `0,8153`, Recall risk `0,6281`, AP `0,8548` (cao hơn AP của LightGBM và XGBoost trong cùng bảng, nhưng Recall thấp hơn LightGBM). Tại 75%, CatBoost đạt Accuracy `0,8750`, Recall `0,7018`, AP `0,9019`; XGBoost tương ứng `0,8779`, `0,7078`, `0,9002`. CatBoost dẫn theo AP trong ba cấu hình boosting thăm dò ở hai mốc này, **không** dẫn Accuracy/Recall; thời gian fit tại hai mốc khoảng 16–18 giây, dài hơn LightGBM/XGBoost dưới một giây trên máy này. Đây là validation, không phải test mới hay bằng chứng tối ưu toàn cục.